# Creating Convokit Corpus element
according to https://github.com/CornellNLP/ConvoKit/blob/master/examples/converting_movie_corpus.ipynb

In [12]:
from convokit import Corpus, Speaker, Utterance
import pandas as pd
import tqdm

In [3]:
media_sum_path = "data/MediaSum/news_dialogue.json"
media_sum_json = pd.read_json(media_sum_path)

In [3]:
media_sum_json

,id,program,date,url,title,summary,utt,speaker
0,NPR-1,News & Notes,2007-11-28,https://www.npr.org/templates/story/story.php?...,Black Actors Give Bible Star Appeal,"More than 400 black actors, artists and minist...","[Now, moving on, Forest Whitaker as Moses, Tis...","[FARAI CHIDEYA, host, FARAI CHIDEYA, host, Mr...."
1,NPR-2,Weekend Edition Sunday,2016-10-23,https://www.npr.org/2016/10/23/499042298/young...,"Young, First-Time Voters Share Views On Electi...",NPR's Rachel Martin speaks with young voters w...,[You have heard it again and again - this is a...,"[RACHEL MARTIN, HOST, ASHANTI MARTINEZ, LAUREN..."
2,NPR-3,News & Notes,2007-11-30,https://www.npr.org/templates/story/story.php?...,Snapshots: On Solid Ground,"In this week's snapshot, actor and playwright ...","[I came close to running out of luck, when I a...","[Mr. JEFF OBAFEMI CARR (Actor, Playwright), CH..."
3,NPR-4,News & Notes,2007-11-30,https://www.npr.org/templates/story/story.php?...,"Washington, D.C. Facing HIV/AIDS Epidemic",A new study says one in 50 people in the natio...,"[This is NEWS & NOTES. I'm Farai Chideya., In ...","[FARAI CHIDEYA, host, FARAI CHIDEYA, host, Dr...."
4,NPR-5,News & Notes,2007-11-30,https://www.npr.org/templates/story/story.php?...,Coping When AIDS Hits Your Family: Part II,When a family member is diagnosed with HIV/AID...,"[I'm Farai Chideya and this is NEWS & NOTES., ...","[FARAI CHIDEYA, host, FARAI CHIDEYA, host, FAR..."
...,...,...,...,...,...,...,...,...
463591,CNN-414237,CNN NEWSROOM,2020-10-25,http://transcripts.cnn.com/TRANSCRIPTS/2010/25...,NaN,"U.S. Officials: Russia, Iran Have Stolen Voter...",[Welcome back to our viewers in the United Sta...,"[BRUNHUBER, NATASHA CHEN, CNN CORRESPONDENT, W..."
463592,CNN-414238,CNN NEWSROOM,2020-10-25,http://transcripts.cnn.com/TRANSCRIPTS/2010/25...,NaN,Nigerian Police Force Mobilize To Quell Worst ...,"[In Nigeria, chaotic scenes of looting and des...","[BRUNHUBER, BRUNHUBER (voice-over), BRUNHUBER ..."
463593,CNN-414239,CNN NEWSROOM,2020-10-25,http://transcripts.cnn.com/TRANSCRIPTS/2010/25...,NaN,COVID-19 Triggers Rise In Asian American Unemp...,[Officials in the U.S. are worried about wides...,"[BRUNHUBER, AMARA WALKER, CNN ANCHOR (voice-ov..."
463594,CNN-414240,STATE OF THE UNION,2020-10-25,http://transcripts.cnn.com/TRANSCRIPTS/2010/25...,NaN,COVID-19 Outbreak Hits Vice President Pence's ...,[Dark winter? U.S. COVID cases hit a new daily...,"[JAKE TAPPER, CNN HOST (voice-over), DONALD TR..."


## 1. Create speakers

**Note**: In the speaker list, authors sometimes have non-unique identifiers (e.g., ‘STEVE PROFFITT’, ‘PROFFITT’ or ‘S. PROFFITT’ refer to the same speaker). See example below. Currently I **do not** address this. I will count each unique identifier as a different speaker. Plus, I will count an identifier that is the same in one conversation as in another as the same speaker in another conversation. This might be incorrect for cases like below with 'UNIDENTIFIED MALE' or 'UNIDENTIFIED FEMALE', but I will not address this for now.

In [43]:
media_sum_json["speaker"][300000]

['CUOMO',
 'ED LAVANDERA, CNN CORRESPONDENT',
 'LAVANDERA (voice-over)',
 'ERIC HOLDER, U.S. ATTORNEY GENERAL',
 'LAVANDERA',
 'UNIDENTIFIED FEMALE',
 'UNIDENTIFIED MALE',
 'UNIDENTIFIED MALE',
 'LAVANDERA',
 'HOLDER',
 'LAVANDERA',
 'LAVANDERA',
 'PEREIRA',
 'PASTOR ROBERT WHITE, PEACE OF MIND CHURCH OF HAPPINESS',
 'PEREIRA',
 'MO IVORY, ATTORNEY/RADIO PERSONALITY',
 'PEREIRA',
 'IVORY',
 'PEREIRA',
 'IVORY',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'IVORY',
 'WHITE',
 'IVORY',
 'PEREIRA',
 'IVORY',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'WHITE',
 'PEREIRA',
 'CUOMO',
 'BERMAN']

I use the incorrect **assumption that each element in the speaker list is a string that is the only unique string for this speaker across the whole dataset**.

In [6]:
# get all speakers from the speaker column
speakers = media_sum_json['speaker']
unique_speakers = sorted(set(name for sublist in speakers for name in sublist))

I create a speaker object that only includes the speaker name as information and identifier.

In [8]:
corpus_speakers = {speaker_name: Speaker(id = speaker_name, meta ={'name': speaker_name}) for speaker_name in unique_speakers}

In [10]:
corpus_speakers['LAVANDERA']

Speaker({'obj_type': 'speaker', 'vectors': [], 'owner': None, 'id': 'LAVANDERA', 'temp_backend': {}, 'meta': {'name': 'LAVANDERA'}})

In [11]:
corpus_speakers['ED LAVANDERA, CNN CORRESPONDENT']

Speaker({'obj_type': 'speaker', 'vectors': [], 'owner': None, 'id': 'ED LAVANDERA, CNN CORRESPONDENT', 'temp_backend': {}, 'meta': {'name': 'ED LAVANDERA, CNN CORRESPONDENT'}})

## 2. Creating utterance objects

In [14]:
type(media_sum_json['utt'][0])

list

In [23]:
utterance_corpus = {}
conversation_meta = {}

count = 0
# iterate over each row in the dataframe
for index, row in tqdm.tqdm(media_sum_json.iterrows(), total=media_sum_json.shape[0]):
    # get the conversation id
    conversation_id = row['id']
    program = row['program']
    date = row['date']
    summary = row['summary']
    url = row['url']
    title = row['title']

    conversation_meta[conversation_id] = {
        'program': program,
        'date': date,
        'summary': summary,
        'url': url,
        'title': title,
        'broadcaster': conversation_id.split('-')[0],  # should be either NPR or CNN
    }

    # get utterance information
    utterance_list = row['utt']
    speaker_list = row['speaker']

    for i, utt in enumerate(utterance_list):
        # create a unique identifier for the utterance as in https://aclanthology.org/2024.emnlp-main.52.pdf
        #   i.e., from the code base ID of the form 'CNN-67148-13' where 'CNN-67148' is the identifier as used in MediaSum and 13 is the index of the utterance in the original utterance list
        utterance_id = f"{conversation_id}-{i}"
        utt_speaker = corpus_speakers[speaker_list[i]]
        utt_text = utt
        reply_to = None if i == 0 else f"{conversation_id}-{i-1}"  # reply_to is None for the first utterance in the conversation
        # timestamp is not provided

        utterance_corpus[utterance_id] = Utterance(
            id=utterance_id,
            speaker=utt_speaker,
            conversation_id=conversation_id,
            reply_to=reply_to,
            text=utt_text,
        )



print(f"Total number of utterances: {len(utterance_corpus)}")

100%|██████████| 463596/463596 [01:01<00:00, 7534.51it/s] 

Total number of utterances: 13919244


In [25]:
# example utterance
utterance_corpus['CNN-67148-13']

Utterance({'obj_type': 'utterance', 'vectors': [], 'speaker_': Speaker({'obj_type': 'speaker', 'vectors': [], 'owner': None, 'id': 'CLARK', 'temp_backend': {}, 'meta': {'name': 'CLARK'}}), 'owner': None, 'id': 'CNN-67148-13', 'temp_backend': {'speaker_id': 'CLARK', 'conversation_id': 'CNN-67148', 'reply_to': 'CNN-67148-12', 'timestamp': None, 'text': "Well, I don't think -- as far as I know, we're not paying anything to Saudi Arabia, for example, right now. In fact, they're still buying weapons. They are having economic difficulties, but they do have oil. But the other countries in the region are in one way or another in financial trouble, and have been for a long time. They've been sustained on a diet of expectations of economic growth, funded by taking short and long term loans that come from commercial banks, sometimes guaranteed by governments. And then they have to repay these loans. And repaying these loans consumes their foreign exchange earnings from their exports and from remi

In [ ]:
utterance_corpus["NPR-1"]

## 3. Creating corpus from list of utterances

In [27]:
utterance_list = utterance_corpus.values()

In [ ]:
media_sum_corpus = Corpus(utterances=utterance_list)

In [ ]:
print("number of conversations in the dataset = {}".format(len(media_sum_corpus.get_conversation_ids())))

In [ ]:
convo_ids = media_sum_corpus.get_conversation_ids()
for i, convo_idx in enumerate(convo_ids[0:5]):
    print("sample conversation {}:".format(i))
    print(media_sum_corpus.get_conversation(convo_idx).get_utterance_ids())

## 4. Updating Conversation and Corpus level metadata

In [ ]:
for convo in media_sum_corpus.iter_conversations():
    # get the conversation id by checking from utterance info
    convo_id = convo.get_id()

    # update meta with additional conversation information
    convo.meta.update(conversation_meta[convo_id])

In [ ]:
media_sum_corpus.get_conversation("CNN-67148").meta

In [ ]:
media_sum_corpus.meta['name'] = 'MediaSum Corpus'

## 5. Adding Paraphrase annotations